# Clase 4 — Buenas prácticas de entrenamiento

Curso "Fundamentos de TensorFlow desde cero" — DIH DATAlife

## Objetivos de esta clase

- Detectar sobreajuste (overfitting) e infra-ajuste (underfitting) a partir de las curvas de entrenamiento y validación
- Aplicar técnicas de regularización: dropout, regularización L2 y early stopping
- Guardar y cargar un modelo entrenado y sus pesos para reutilizarlo

Trabajamos con el dataset IMDB (Internet Movie Database) de reseñas de cine, ya incluido en Keras, porque es el mismo ejemplo que usa el material de la certificación EITC/AI/TFF (European IT Certificate / Artificial Intelligence / TensorFlow Fundamentals) para introducir el sobreajuste.

## 1. Sobreajuste e infra-ajuste

- **Sobreajuste:** la pérdida (loss, el error del modelo) de entrenamiento sigue bajando, pero la de validación se estanca o sube. El modelo memoriza en vez de generalizar
- **Infra-ajuste:** las dos pérdidas se quedan altas. El modelo no tiene capacidad o entrenamiento suficiente

Para verlo con nuestros propios ojos, vamos a entrenar tres modelos con distinta capacidad (número de neuronas) sobre el mismo problema y comparar sus curvas.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, regularizers
import matplotlib.pyplot as plt

print("Versión de TensorFlow:", tf.__version__)

### Preparamos los datos

El dataset IMDB contiene 50 000 reseñas de películas etiquetadas como positivas o negativas. Cada reseña llega ya convertida en una lista de números: cada número es el ID de una palabra (el ID 1 es la palabra más frecuente, el ID 2 la segunda más frecuente, y así seguido).

Para poder meter esas reseñas en una red densa, las convertimos en un vector "multi-hot" de 10 000 posiciones (una por cada palabra del vocabulario): la posición vale 1.0 si la palabra aparece en la reseña, 0.0 si no aparece. Usamos un subconjunto de los datos para que el entrenamiento sea rápido durante la clase.

In [ ]:
NUM_WORDS = 10000
N_TRAIN = 10000
N_VAL = 5000

(train_data, train_labels), (test_data, test_labels) = tf.keras.datasets.imdb.load_data(num_words=NUM_WORDS)

def multi_hot_sequences(sequences, dimension):
    """Convierte cada reseña (lista de IDs de palabra) en un vector de
    'dimension' posiciones: 1.0 si la palabra aparece, 0.0 si no aparece."""
    multi_hot = np.zeros((len(sequences), dimension), dtype="float32")
    for i, word_indices in enumerate(sequences):
        multi_hot[i, word_indices] = 1.0
    return multi_hot

x_train = multi_hot_sequences(train_data[:N_TRAIN], NUM_WORDS)
y_train = np.asarray(train_labels[:N_TRAIN]).astype("float32")

x_val = multi_hot_sequences(test_data[:N_VAL], NUM_WORDS)
y_val = np.asarray(test_labels[:N_VAL]).astype("float32")

print("Forma de x_train:", x_train.shape)
print("Forma de x_val:", x_val.shape)

# Así se ve una reseña ya codificada: casi todo ceros, con algunos unos
# en las posiciones de las palabras (de baja ID, las más frecuentes) que contiene
plt.figure(figsize=(9, 2))
plt.plot(x_train[0])
plt.xlabel("ID de palabra")
plt.ylabel("¿Aparece en la reseña?")
plt.title("Codificación multi-hot de la primera reseña de entrenamiento")
plt.show()

### Tres modelos, tres capacidades distintas

- **baseline:** dos capas densas de 16 neuronas — un tamaño razonable
- **small:** dos capas densas de cuatro neuronas — le falta capacidad
- **big:** dos capas densas de 512 neuronas — le sobra capacidad

Los tres terminan en una neurona con activación sigmoide, porque es una clasificación binaria (reseña positiva o negativa).

In [ ]:
def build_baseline_model():
    return tf.keras.Sequential([
        layers.Dense(16, activation="relu", input_shape=(NUM_WORDS,)),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])

def build_small_model():
    return tf.keras.Sequential([
        layers.Dense(4, activation="relu", input_shape=(NUM_WORDS,)),
        layers.Dense(4, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])

def build_big_model():
    return tf.keras.Sequential([
        layers.Dense(512, activation="relu", input_shape=(NUM_WORDS,)),
        layers.Dense(512, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])


def compile_and_fit(model, max_epochs=20, extra_callbacks=None):
    """Compila y entrena un modelo con early stopping incluido: si la
    pérdida de validación no mejora durante 5 épocas seguidas (patience=5),
    detiene el entrenamiento y recupera los mejores pesos vistos."""
    model.compile(optimizer="adam",
                  loss=tf.keras.losses.BinaryCrossentropy(),
                  metrics=["accuracy"])

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True)

    callbacks = [early_stop] + (extra_callbacks or [])

    history = model.fit(x_train, y_train,
                         epochs=max_epochs,
                         batch_size=512,
                         validation_data=(x_val, y_val),
                         callbacks=callbacks,
                         verbose=0)
    return history

In [ ]:
histories = {}
histories["baseline"] = compile_and_fit(build_baseline_model())
histories["small"] = compile_and_fit(build_small_model())
histories["big"] = compile_and_fit(build_big_model())

for name, history in histories.items():
    print(f"{name}: entrenó {len(history.epoch)} épocas (máximo 20, gracias a early stopping)")

In [ ]:
def plot_histories(histories_dict, metric="loss"):
    plt.figure(figsize=(9, 6))
    for name, history in histories_dict.items():
        epochs = range(1, len(history.history[metric]) + 1)
        linea = plt.plot(epochs, history.history[metric],
                          label=f"{name} (entrenamiento)")
        plt.plot(epochs, history.history["val_" + metric], "--",
                  color=linea[0].get_color(), label=f"{name} (validación)")
    plt.xlabel("Época")
    plt.ylabel(metric)
    plt.legend()
    plt.show()

plot_histories(histories)

Fíjate en el patrón:

- **small** tarda más en bajar su pérdida de entrenamiento: le cuesta aprender (infra-ajuste leve)
- **baseline** baja bien las dos curvas y se mantienen razonablemente cerca
- **big** baja muy rápido su pérdida de entrenamiento, pero su pérdida de validación se separa pronto y empieza a subir: sobreajuste claro

## 2. Regularización: L2 y dropout

Vamos a coger el modelo **big**, que es el que peor generaliza, y aplicarle dos técnicas de regularización a la vez:

- **L2** (`kernel_regularizer=regularizers.l2(0.001)`): penaliza los pesos grandes en la función de pérdida
- **Dropout** (`layers.Dropout(0.5)`): apaga al azar el 50% de las neuronas de la capa anterior en cada paso de entrenamiento

In [ ]:
def build_regularized_model():
    return tf.keras.Sequential([
        layers.Dense(512, activation="relu",
                     kernel_regularizer=regularizers.l2(0.001),
                     input_shape=(NUM_WORDS,)),
        layers.Dropout(0.5),
        layers.Dense(512, activation="relu",
                     kernel_regularizer=regularizers.l2(0.001)),
        layers.Dropout(0.5),
        layers.Dense(1, activation="sigmoid"),
    ])

histories["big_regularizado"] = compile_and_fit(build_regularized_model())

plot_histories({"big": histories["big"], "big_regularizado": histories["big_regularizado"]})

La versión regularizada tarda más en sobreajustar (o no llega a hacerlo dentro del número de épocas que le dejamos) y su pérdida de validación se queda más cerca de la de entrenamiento. Ese hueco entre las dos curvas es exactamente lo que queremos reducir.

## 3. Early stopping en acción

Ya lo hemos usado sin comentarlo: la función `compile_and_fit` incluye un callback `EarlyStopping` que vigila `val_loss` y para el entrenamiento si no mejora durante cinco épocas seguidas, devolviendo los mejores pesos vistos (`restore_best_weights=True`). Fíjate en el número de épocas que entrenó cada modelo en la celda de arriba: casi ninguno llegó a las 20 épocas del límite.

Sin early stopping tendrías que decidir a mano cuántas épocas entrenar, y el riesgo de pasarte y sobreajustar es alto.

## 4. Guardar y cargar un modelo

Ahora que tenemos un modelo entrenado, vamos a guardarlo de dos formas distintas: guardando solo los pesos con un checkpoint, y guardando el modelo completo.

In [ ]:
# Entrenamos un modelo "baseline" de nuevo, esta vez guardando sus pesos
# en un checkpoint cada vez que mejora la pérdida de validación
modelo = build_baseline_model()
modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

checkpoint_path = "checkpoint_pesos.weights.h5"
cp_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_weights_only=True,
    save_best_only=True,
    monitor="val_loss",
    verbose=0,
)

modelo.fit(x_train, y_train, epochs=10, batch_size=512,
           validation_data=(x_val, y_val),
           callbacks=[cp_callback], verbose=0)

loss, acc = modelo.evaluate(x_val, y_val, verbose=0)
print(f"Precisión del modelo recién entrenado: {acc:.2%}")

In [ ]:
# Reconstruimos la MISMA arquitectura sin entrenar: su precisión debería
# rondar el 50% (equivalente a lanzar una moneda al aire)
modelo_sin_entrenar = build_baseline_model()
modelo_sin_entrenar.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

loss, acc = modelo_sin_entrenar.evaluate(x_val, y_val, verbose=0)
print(f"Precisión sin entrenar: {acc:.2%}")

# Cargamos en esa misma arquitectura los pesos que guardamos antes
modelo_sin_entrenar.load_weights(checkpoint_path)

loss, acc = modelo_sin_entrenar.evaluate(x_val, y_val, verbose=0)
print(f"Precisión tras cargar los pesos guardados: {acc:.2%}")

La precisión coincide (o casi) con la del modelo original: hemos recuperado el conocimiento aprendido sin volver a entrenar. Esto solo funciona porque `modelo_sin_entrenar` tiene exactamente la misma arquitectura que el modelo que guardamos los pesos.

### Guardar el modelo completo

Si en lugar de los pesos quieres guardar también la arquitectura y la configuración del optimizador, usa `model.save()` con el formato `.keras`, el recomendado en TensorFlow 2.x.

In [ ]:
modelo.save("modelo_clase04.keras")

# Cargamos el modelo completo en una variable nueva, sin necesidad de
# volver a definir la arquitectura en código
modelo_cargado = tf.keras.models.load_model("modelo_clase04.keras")
modelo_cargado.summary()

loss, acc = modelo_cargado.evaluate(x_val, y_val, verbose=0)
print(f"Precisión del modelo cargado desde 'modelo_clase04.keras': {acc:.2%}")

## Resumen

- Compara la pérdida de entrenamiento y de validación en cada época para detectar sobreajuste (se separan) o infra-ajuste (las dos se quedan altas)
- Dropout, regularización L2 y early stopping son las técnicas más directas contra el sobreajuste, y combinarlas suele funcionar mejor que usar una sola
- `ModelCheckpoint` + `save_weights` / `load_weights` guardan solo los pesos: necesitas reconstruir la misma arquitectura para reutilizarlos
- `model.save("archivo.keras")` + `tf.keras.models.load_model(...)` guardan y recuperan el modelo completo, listo para usar

Sigue practicando en el notebook `clase04_practica_buenas_practicas_entrenamiento.ipynb` y consulta el capítulo 06 del manual para repasar la teoría con más detalle.